In [4]:
#importing libraries
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

INPUT_FILE = "coverage_results_rich.json"

# GLOBAL FONT SETTINGS
AXIS_LABEL_SIZE = 18
TITLE_SIZE = 20
TICK_SIZE = 14
LEGEND_SIZE = 14

# LOADING RESULTS
with open(INPUT_FILE, "r") as f:
    payload = json.load(f)

if isinstance(payload, list):
    data = payload
else:
    data = payload["dataset_results"]

df = pd.DataFrame(data)

print("\n✅ Loaded semantic completeness results")
print(df.head())

# MAIN COLUMNS (t = 0.7)

title_col = "title_coverage_0.7"
description_col = "description_coverage_0.7"


# SUMMARY STATISTICS

print("\n📊 SUMMARY STATISTICS (t=0.7)\n")

summary = {
    "Metric": ["Mean", "Median", "Std", "Min", "Max"],

    "Title Coverage (0.7)": [
        df[title_col].mean(),
        df[title_col].median(),
        df[title_col].std(),
        df[title_col].min(),
        df[title_col].max(),
    ],

    "Description Coverage (0.7)": [
        df[description_col].mean(),
        df[description_col].median(),
        df[description_col].std(),
        df[description_col].min(),
        df[description_col].max(),
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df)


# BOXPLOT

plt.figure(figsize=(8, 6))

box_data = [
    df[title_col],
    df[description_col]
]

labels = [
    "Title",
    "Description"
]

bp = plt.boxplot(
    box_data,
    tick_labels=labels,
    patch_artist=True
)

# DIFFERENT COLORS FOR TITLE & DESCRIPTION
colors = [
    "steelblue",
    "darkorange"
]

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

medians = [
    df[title_col].median(),
    df[description_col].median()
]

for i, median in enumerate(medians, start=1):
    plt.text(
        i + 0.08,
        median,
        f"{median:.2f}",
        verticalalignment='center',
        fontsize=10
    )

plt.ylabel(
    "Semantic Completeness",
    fontsize=AXIS_LABEL_SIZE
)

plt.title(
    "Semantic Completeness Comparison ($\\tau$=0.7)",
    fontsize=TITLE_SIZE
)

plt.xticks(fontsize=TICK_SIZE)
plt.yticks(fontsize=TICK_SIZE)

plt.tight_layout()

plt.savefig(
    "coverage_boxplot.pdf",
    bbox_inches='tight',
    pad_inches=0.2
)

plt.close()


# RANGE DISTRIBUTION

print("\n📊 SEMANTIC COMPLETENESS RANGE DISTRIBUTION (0–1)\n")

bins = [i/10 for i in range(11)]

labels = [
    f"{bins[i]:.1f}-{bins[i+1]:.1f}"
    for i in range(len(bins)-1)
]

df["title_coverage_bin"] = pd.cut(
    df[title_col],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df["description_coverage_bin"] = pd.cut(
    df[description_col],
    bins=bins,
    labels=labels,
    include_lowest=True
)

title_bin_counts = (
    df["title_coverage_bin"]
    .value_counts()
    .sort_index()
)

description_bin_counts = (
    df["description_coverage_bin"]
    .value_counts()
    .sort_index()
)

title_bin_percent = (title_bin_counts / len(df)) * 100
description_bin_percent = (description_bin_counts / len(df)) * 100

range_df = pd.DataFrame({
    "Range": title_bin_counts.index,
    "Title Percentage": title_bin_percent.values,
    "Description Percentage": description_bin_percent.values
})

print(range_df)


# RANGE BAR PLOT

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(
    x - width/2,
    title_bin_percent.values,
    width,
    label="Title"
)

plt.bar(
    x + width/2,
    description_bin_percent.values,
    width,
    label="Description"
)

plt.xticks(
    x,
    labels,
    rotation=45,
    fontsize=TICK_SIZE
)

plt.yticks(fontsize=TICK_SIZE)

plt.xlabel(
    "Semantic Completeness Range",
    fontsize=AXIS_LABEL_SIZE
)

plt.ylabel(
    "Percentage of Datasets",
    fontsize=AXIS_LABEL_SIZE
)

plt.title(
    "Semantic Completeness Distribution (0–1 Range)",
    fontsize=TITLE_SIZE
)

plt.legend(fontsize=LEGEND_SIZE)

plt.tight_layout()

plt.savefig(
    "coverage_range_distribution.pdf",
    bbox_inches="tight",
    pad_inches=0.2
)

plt.close()


# MULTI-THRESHOLD COMPARISON

print("\n📊 MULTI-THRESHOLD SEMANTIC COMPLETENESS COMPARISON\n")

title_threshold_means = df[[
    "title_coverage_0.6",
    "title_coverage_0.7",
    "title_coverage_0.8"
]].mean()

description_threshold_means = df[[
    "description_coverage_0.6",
    "description_coverage_0.7",
    "description_coverage_0.8"
]].mean()

print("\nTitle Semantic Completeness Means\n")
print(title_threshold_means)

print("\nDescription Semantic Completeness Means\n")
print(description_threshold_means)

robustness_df = pd.DataFrame({
    "Threshold": [0.6, 0.7, 0.8],
    "Title Semantic Completeness Mean": title_threshold_means.values,
    "Description Semantic Completeness Mean": description_threshold_means.values
})

print("\n📊 ROBUSTNESS SUMMARY TABLE\n")
print(robustness_df)


# MEAN SEMANTIC COMPLETENESS BY KEYWORD COUNT GROUP

print("\n📊 SEMANTIC COMPLETENESS BY KEYWORD COUNT GROUP\n")

# Keyword count buckets
df["keyword_group"] = pd.cut(
    df["num_keywords"],
    bins=[0, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

# Mean completeness per bucket
title_group_means = (
    df.groupby("keyword_group")["title_coverage_0.7"]
    .mean()
)

description_group_means = (
    df.groupby("keyword_group")["description_coverage_0.7"]
    .mean()
)

print("\nTitle Semantic Completeness by Keyword Group\n")
print(title_group_means)

print("\nDescription Semantic Completeness by Keyword Group\n")
print(description_group_means)

# PLOT

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
    sharey=True
)

# TITLE

axes[0].plot(
    title_group_means.index.astype(str),
    title_group_means.values,
    marker="o",
    linewidth=2,
    color="steelblue"
)

axes[0].set_title(
    "Title Semantic Completeness",
    fontsize=TITLE_SIZE
)

axes[0].set_xlabel(
    "Keyword Count Group",
    fontsize=AXIS_LABEL_SIZE
)

axes[0].set_ylabel(
    "Mean Semantic Completeness",
    fontsize=AXIS_LABEL_SIZE
)

axes[0].tick_params(
    axis='both',
    labelsize=TICK_SIZE
)

axes[0].set_ylim(0, 1)

axes[0].grid(
    linestyle="--",
    alpha=0.4
)

# DESCRIPTION

axes[1].plot(
    description_group_means.index.astype(str),
    description_group_means.values,
    marker="o",
    linewidth=2,
    color="darkorange"
)

axes[1].set_title(
    "Description Semantic Completeness",
    fontsize=TITLE_SIZE
)

axes[1].set_xlabel(
    "Keyword Count Group",
    fontsize=AXIS_LABEL_SIZE
)

axes[1].tick_params(
    axis='both',
    labelsize=TICK_SIZE
)

axes[1].set_ylim(0, 1)

axes[1].grid(
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()

plt.savefig(
    "completeness_by_keyword_group.pdf",
    bbox_inches="tight",
    pad_inches=0.2
)

plt.close()

# CORRELATION

print("\n🔗 CORRELATION MATRIX\n")

correlation = df[[
    "title_coverage_0.7",
    "description_coverage_0.7",
    "avg_title_max_similarity",
    "avg_description_max_similarity",
    "num_keywords",
    "num_title_concepts",
    "num_description_concepts"
]].corr()

print(correlation)


# TOP & BOTTOM DATASETS

top_title = df.sort_values(
    title_col,
    ascending=False
).head(10)

bottom_title = df.sort_values(
    title_col,
    ascending=True
).head(10)

top_description = df.sort_values(
    description_col,
    ascending=False
).head(10)

bottom_description = df.sort_values(
    description_col,
    ascending=True
).head(10)

print("\n🏆 TOP 10 TITLE SEMANTIC COMPLETENESS DATASETS\n")
print(top_title)

print("\n⚠️ BOTTOM 10 TITLE SEMANTIC COMPLETENESS DATASETS\n")
print(bottom_title)

print("\n🏆 TOP 10 DESCRIPTION SEMANTIC COMPLETENESS DATASETS\n")
print(top_description)

print("\n⚠️ BOTTOM 10 DESCRIPTION SEMANTIC COMPLETENESS DATASETS\n")
print(bottom_description)


# SAVE TABLES

summary_df.to_csv("coverage_summary_statistics.csv", index=False)
range_df.to_csv("coverage_range_distribution.csv", index=False)
robustness_df.to_csv("coverage_robustness_summary.csv", index=False)

print("\n✅ Coverage analysis complete!")


✅ Loaded semantic completeness results
                                          dataset_id  num_keywords  \
0  http://data.europa.eu/88u/dataset/taxi-and-pri...            12   
1  http://data.europa.eu/88u/dataset/qics-data-2d...             6   
2  http://data.europa.eu/88u/dataset/free-school-...             9   
3  http://data.europa.eu/88u/dataset/farm-census-...            14   
4  http://data.europa.eu/88u/dataset/register-of-...             9   

   num_title_concepts  num_description_concepts  avg_title_max_similarity  \
0                   5                         5                  0.846611   
1                   5                         5                  0.558002   
2                   5                         5                  0.789732   
3                   5                         5                  0.728969   
4                   5                         5                  0.808039   

   avg_description_max_similarity  title_coverage_0.6  title_coverage_0.7  \

C:\Users\deper\AppData\Local\Temp\ipykernel_23992\2621756928.py:289: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("keyword_group")["title_coverage_0.7"]
C:\Users\deper\AppData\Local\Temp\ipykernel_23992\2621756928.py:294: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("keyword_group")["description_coverage_0.7"]



🔗 CORRELATION MATRIX

                                title_coverage_0.7  description_coverage_0.7  \
title_coverage_0.7                        1.000000                  0.532475   
description_coverage_0.7                  0.532475                  1.000000   
avg_title_max_similarity                  0.897266                  0.503656   
avg_description_max_similarity            0.537997                  0.854682   
num_keywords                              0.413183                  0.320977   
num_title_concepts                        0.090606                 -0.027041   
num_description_concepts                  0.001312                  0.058999   

                                avg_title_max_similarity  \
title_coverage_0.7                              0.897266   
description_coverage_0.7                        0.503656   
avg_title_max_similarity                        1.000000   
avg_description_max_similarity                  0.599360   
num_keywords                        